In [ ]:
# ==================== 🚨 升级版：暴力清洗与匹配 🚨 ====================
print("读取人口数据...")
df_pop_raw = pd.read_csv(r"/path/to/data/Population by Age and Sex - US, States, Counties.csv")

# 1. 提取2025年县级数据
df_pop_2025 = df_pop_raw[(df_pop_raw['Year'] == 2025) & (df_pop_raw['Countyfips'] > 0)].copy()

# 2. 拆分名字
split_desc = df_pop_2025['Description'].str.split(', ', expand=True)
df_pop_2025['County_Name'] = split_desc[0]
df_pop_2025['State_Abbr'] = split_desc[1].str.strip()

# 3. 定义一个超级清洗函数
def clean_name(name):
    if pd.isna(name): return name
    name = str(name).upper().strip()
    # 干掉所有乱七八糟的后缀
    for suffix in [' COUNTY', ' PARISH', ' BOROUGH', ' CENSUS AREA', ' CITY', ' MUNICIPALITY']:
        name = name.replace(suffix, '')
    # 解决缩写问题
    name = name.replace('ST. ', 'SAINT ').replace('FT. ', 'FORT ').replace('MT. ', 'MOUNT ')
    return name.strip()

df_pop_2025['County_Clean'] = df_pop_2025['County_Name'].apply(clean_name)

# 4. 州名映射（直接复制）
state_map = {'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming', 'DC': 'District of Columbia', 'PR': 'Puerto Rico'}
df_pop_2025['State_Name'] = df_pop_2025['State_Abbr'].map(state_map)

# 提取最终列
df_pop_clean = df_pop_2025[['State_Name', 'County_Clean', 'Total Population']].copy()
df_pop_clean.columns = ['State', 'County', 'Population']

# 对边界数据同样清洗
gdf_boundary['NAME_1_clean'] = gdf_boundary['NAME_1'].str.upper().str.strip()
gdf_boundary['NAME_2_clean'] = gdf_boundary['NAME_2'].apply(clean_name)

# 合并
gdf_merged = gdf_boundary.merge(
    df_pop_clean, 
    left_on=['NAME_1_clean', 'NAME_2_clean'], 
    right_on=['State', 'County'], 
    how='left'
)

print(f"升级清洗后，匹配到的县级数据量: {gdf_merged['Population'].notna().sum()} / {len(gdf_merged)}")
# （如果这里输出接近 3100，说明基本全匹配上了！）

# ==================== 🚨 升级版：防发白极值渲染 🚨 ====================
print("开始批量出图（使用分位数防发白）...")
output_dir = r"/path/to/output/Pop_Maps"
os.makedirs(output_dir, exist_ok=True)

for state_name, state_data in gdf_merged.groupby('NAME_1'):
    if state_data['Population'].isna().all():
        continue
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    # 关键点：scheme='Quantiles' (分位数分级)，k=5 (分5档)
    state_data.plot(
        column='Population', 
        cmap='Reds', 
        scheme='Quantiles', # 这个参数直接解决地图全白的问题
        k=5,
        linewidth=0.5, 
        ax=ax, 
        edgecolor='0.5',
        legend=True,
        legend_kwds={'title': "Pop 2025", 'loc': 'lower right'}
    )
    
    ax.set_title(f"{state_name} Population (Quantiles)", fontsize=16)
    ax.set_axis_off()
    
    plt.savefig(f"{output_dir}/{state_name}_pop.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

print("🎉 重生版流水线跑完！去看看这次的地图是不是全彩了！")